# CARE Patch Generation (Real Data, Nested Layout)

This notebook generates training patches (`.npz`) for CARE / csbdeep training from *real* microscopy datasets stored in a nested folder structure. **Specifically for data from ISS preprocessing pipeline**. 

### What it does

For each detected *sample folder*, the pipeline:

1. Finds matching **source–target image pairs** by relative path  
   (e.g. `raw/.../file.tif` ↔ `rlf50/.../file.tif`)

2. Splits pairs into two datasets:
   - **NON_DAPI**: all channels except the specified DAPI channel  
   - **DAPI_ONLY**: only the DAPI channel  

3. Excludes files without a `_chN.tif` channel pattern  

4. Applies image-level filtering to remove:
   - empty / constant images  
   - corrupted data (NaN / Inf / extreme values)  
   - mismatched or low-information pairs  

5. Creates training patches using `csbdeep.data.create_patches(...)`  
   and saves them as `.npz` files

   
### Expected folder layout (per sample)

A *sample folder* is any directory that contains both:
- `SOURCE_DIRNAME`
- `TARGET_DIRNAME`

Structure:

<CARE_ROOT>/<any_structure>/<sample>/
├── SOURCE_DIRNAME/
│   └── R*/preprocessing/Cycle*/4_retiled/*.tif
└── TARGET_DIRNAME/
    └── R*/preprocessing/Cycle*/4_retiled/*.tif

Matching is done by **relative path**, for example:

raw/.../Cycle1_s0_ch0.tif  
rlf50/.../Cycle1_s0_ch0.tif

### Outputs

Per sample (saved in `<sample>/train_patches/`):

- `<group>__<sample>__NON_DAPI__train_patches.npz`
- `<group>__<sample>__DAPI_ONLY__train_patches.npz`

Optional merged outputs (saved in `<CARE_ROOT>/train_patches/`):

- `ALL_SAMPLES__NON_DAPI__train_patches.npz`
- `ALL_SAMPLES__DAPI_ONLY__train_patches.npz`

Each `.npz` file contains:
- `X`: input patches  
- `Y`: target patches  
- `axes`: axis specification  

### Next step

Use the generated `.npz` files for CARE training:
- train main model on **NON_DAPI**
- optionally train a separate model on **DAPI_ONLY**

## Imports

In [ ]:
from pathlib import Path

from ISS_CARE.ISS_CARE_datagen import (
    run_patch_generation,
    visualize_saved_patches_across_samples,
)

## User settings

Edit all user-configurable parameters in the **User settings** cell at the top of the notebook.

### Core paths

- `CARE_ROOT`: root folder containing all CARE training data  
- `CARE_SUBDIRS`: subdirectories to process  
  - `[]` → include **all subdirectories under CARE_ROOT**  
  - or specify manually, e.g. `["sample1", "sample2"]`  

- `SOURCE_DIRNAME`: folder name containing input images (e.g. `"raw"`)  
- `TARGET_DIRNAME`: folder name containing target images (e.g. `"gt"` or `"rlf50"`)  

- `PATTERN`: recursive glob pattern to find image tiles  
  - default: `"**/*.tif"`  

- `DAPI_CHANNEL_INDEX`: 0-based channel index for DAPI (e.g. `4`)  
  - DAPI is handled separately from other channels  

### Sampling (per sample)

- `MAX_IMAGES_PER_SAMPLE` (int | None): max number of image pairs per sample  
  - typical: `100–600`  
- `FRACTION_IMAGES_PER_SAMPLE` (float | None): fraction of pairs to use  
  - e.g. `0.25` = 25%  

Rules:
- If both are set → `MAX_IMAGES_PER_SAMPLE` takes priority  
- If both are `None` → use all images  

### Patch extraction

- `PATCH_SIZE`: patch size  
  - default: `(128, 128)`  

- `N_PATCHES_PER_IMAGE`: patches per image pair  
  - default: `4`  
  - typical: `3–8`  
  - fewer → faster, less redundancy  
  - more → larger dataset but diminishing returns  

- `PATCH_FILTER_THRESHOLD`: filter low-signal patches  
  - default: `0.05`  

  Guidelines:
  - `0.0` → no filtering  
  - `0.01` → light filtering  
  - `0.02–0.05` → good balance  
  - `>0.1` → often too aggressive  

> Image-level filtering is already applied before patching  
> → this threshold only affects **local patch content**


### Axes

- `AXES`: image axes (default: `"YX"`)  
- `PATCH_AXES`: patch axes (default: `"YX"`)  


### Output

- `PATCH_DIRNAME`: output folder for patches  
  - default: `"train_patches"`  

- `MERGE_ALL_SAMPLES` (bool): merge all samples into one dataset  
  - default: `True`  

- `MERGED_NON_DAPI_NAME`: filename for merged non-DAPI dataset  
- `MERGED_DAPI_NAME`: filename for merged DAPI dataset  

### Internal filtering (advanced, usually keep defaults)

These remove bad training pairs automatically:

- `min_source_std = 1e-6`, `min_target_std = 1e-6`  
  → remove constant images  

- `extreme_value_cutoff = 1e6`  
  → remove corrupted images  

- `check_half_plane_artifacts = True`  
  → remove half-black / tiling artifacts  

- `check_signal_consistency = True`  
  → remove mismatched pairs (signal vs empty)  

- `check_low_information_target = True`  
  → remove weak / low-quality targets  

> These are safe defaults and typically should not be changed

In [ ]:
# ----------------------------
# Minimal user settings
# ----------------------------

HOME = Path.home()
CARE_ROOT = HOME / "moldia-archive" / "CARE_training"

# [] = process all subdirectories under CARE_ROOT
CARE_SUBDIRS = ["christina", "Victoria", "maria_e", "yimin"]

SOURCE_DIRNAME = "raw"
TARGET_DIRNAME = "rlf50"
PATTERN = "**/4_retiled/*.tif"
DAPI_CHANNEL_INDEX = 4

# ----------------------------
# Sampling
# ----------------------------

MAX_IMAGES_PER_SAMPLE = None
SAMPLING_SEED = 42

# ----------------------------
# Patch generation
# ----------------------------

AXES = "YX"          # input image axes (2D microscopy → "YX")
PATCH_AXES = "YX"    # patch axes (should match AXES for 2D data)

PATCH_SIZE = (128, 128)
N_PATCHES_PER_IMAGE = 32
PATCH_FILTER_THRESHOLD = 0.05

# ----------------------------
# Output
# ----------------------------

PATCH_DIRNAME = "train_patches"
MERGE_ALL_SAMPLES = True
MERGED_NON_DAPI_NAME = "NON_DAPI_train_patches_Leica_all_005.npz"
MERGED_DAPI_NAME = "DAPI_ONLY_train_patches_Leica_all_005.npz"

## Run patch generation

Run this cell to generate CARE training patches from your data.

This step will:

- automatically detect all valid **sample directories**
- match `SOURCE_DIRNAME` and `TARGET_DIRNAME` images by **relative path**
- optionally **subsample images per sample**
- split data into:
  - **NON_DAPI** (main model)
  - **DAPI_ONLY** (separate model)
- generate training patches using `csbdeep`
- save `.npz` patch files per sample
- optionally create **merged datasets across all samples**
- automatically generate and save **metadata files** with parameters and outputs

Outputs include:

- per-sample patch files in `<sample>/train_patches/`
- matching per-sample metadata files next to each patch file
- optional merged datasets in `<CARE_ROOT>/train_patches/`
- run metadata file:  
  `<CARE_ROOT>/train_patches/patch_generation_metadata.json`

⚠️ Depending on dataset size, this step can take time.

In [ ]:
results = run_patch_generation(
    # ----------------------------
    # Paths and dataset
    # ----------------------------
    care_root=CARE_ROOT,
    care_subdirs=CARE_SUBDIRS,
    source_dirname=SOURCE_DIRNAME,
    target_dirname=TARGET_DIRNAME,
    pattern=PATTERN,
    dapi_channel_index=DAPI_CHANNEL_INDEX,

    # ----------------------------
    # Patch generation
    # ----------------------------
    axes=AXES,
    patch_size=PATCH_SIZE,
    n_patches_per_image=N_PATCHES_PER_IMAGE,
    patch_axes=PATCH_AXES,
    patch_filter_threshold=PATCH_FILTER_THRESHOLD,
    patch_dirname=PATCH_DIRNAME,

    # ----------------------------
    # Sampling
    # ----------------------------
    max_images_per_sample=MAX_IMAGES_PER_SAMPLE,
    sampling_seed=SAMPLING_SEED,

    # ----------------------------
    # Output
    # ----------------------------
    merge_all_samples=MERGE_ALL_SAMPLES,
    merged_non_dapi_name=MERGED_NON_DAPI_NAME,
    merged_dapi_name=MERGED_DAPI_NAME,
)

In [ ]:
print("Metadata file:", results["metadata_file"])

if results["merged_non_dapi_file"]:
    print("Merged NON_DAPI file:", results["merged_non_dapi_file"])

if results["merged_dapi_file"]:
    print("Merged DAPI file:", results["merged_dapi_file"])

## Visualize generated patches

Run this cell to inspect a few example patch pairs from each sample.

This will:

- load saved `.npz` patch files
- randomly select patch pairs per sample
- display **input (top)** and **target (bottom)** images
- optionally apply normalization for better visibility

This is useful to:

- verify that source and target are correctly aligned  
- check patch quality and signal content  
- spot potential issues before training  

In [ ]:
if results["all_patch_files_non_dapi"]:
    visualize_saved_patches_across_samples(
        results["all_patch_files_non_dapi"],
        n_show_per_sample=3,
        variant_name="NON_DAPI",
        random_seed=42,
        normalize=True,
    )